# Further Fine-tune `cardd-yolov8s` — Boundary-Precision Recipe

Continues from the best checkpoint of the earlier `cardd-yolov8s` continuation run (overall mask mAP50 0.36 / mAP50-95 0.165 at its best epoch), with a genuinely new angle rather than repeating what was already tried:

**What was already tried on this model and didn't move `dent`/`scratch`/`crack`:** targeted augmentation (`copy_paste`, rotation, brightness/saturation jitter). Overall mAP improved marginally (+0.005) but the three hard classes stayed flat.

**What this run tries instead:** the `dfl`-raised, `cls`-lowered boundary-precision recipe (`cls=0.3`, `dfl=1.7`) from the published `harpreetsahota/car-dd-segmentation-yolov11` training config — this targets a different mechanism (box/mask boundary sharpness in the loss function itself) than augmentation does, and was never actually tested on this model lineage, since the YOLO11x-seg attempt where it came from turned out too expensive to iterate on properly.

**Everything accumulated from the last several rounds of debugging is reused here:**
- Compute-capability check (Section 1) — catches a GPU/PyTorch-build mismatch immediately, not after building the whole pipeline
- RAM-aware cache decision (Section 4)
- **Corrected two-part probe timing** (Section 5) — training and validation measured as two separate timed runs, not one combined number scaled by the training-set-size ratio. That old approach silently inflated estimates by ~8x (confirmed directly: a real run reported ~177 min/epoch when the true cost was ~21 min/epoch), because it multiplied a roughly-fixed validation cost as if it scaled with training data.
- Multi-session checkpoint relay with **honest backup success/failure reporting** (Section 6), own distinct backup identity so it can never collide with either of the two earlier `cardd-yolov8s` runs' backups

**Prerequisite:** the earlier `cardd-yolov8s` continuation run's backup dataset (`cardd-seg-continuation-checkpoint-backup`) must exist — this notebook downloads its `best.pt` as the starting checkpoint, rather than assuming local files persist across Kaggle sessions.

**Setup on Kaggle:** attach the published segmentation dataset as an input, set the accelerator to **GPU T4**.

## 1. Setup

In [1]:
!pip install -q ultralytics psutil

import json, shutil, time, os, subprocess
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable GPU in Settings → Accelerator before running."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# Compute-capability check -- catches a GPU/PyTorch-build mismatch immediately
# (found the hard way with P100 against this environment's torch/cu128 build).
device_cap = torch.cuda.get_device_capability(0)
sm_str = f"sm_{device_cap[0]}{device_cap[1]}"
supported = torch.cuda.get_arch_list()
print(f"GPU compute capability: {device_cap} ({sm_str})")
print(f"This torch build's compiled kernel architectures: {supported}")

if sm_str not in supported:
    raise RuntimeError(
        f"This GPU's architecture ({sm_str}) has no compiled kernels in the "
        f"installed PyTorch build (supports: {supported}). Fix: change Settings "
        f"-> Accelerator to T4 and re-run from this cell."
    )
print("GPU/PyTorch build compatible.")

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
ram_available_gb = psutil.virtual_memory().available / 1e9
print(f"System RAM: {ram_gb:.1f} GB total, {ram_available_gb:.1f} GB available")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.7 MB/s eta 0:00:00a 0:00:01
GPU: Tesla T4
VRAM: 15.6 GB
GPU compute capability: (7, 5) (sm_75)
This torch build's compiled kernel architectures: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU/PyTorch build compatible.
System RAM: 33.7 GB total, 31.8 GB available


## 2. Mount the converted segmentation dataset

In [2]:
!ls /kaggle/input/

# EDIT if your dataset is mounted at a different path
SEG_ROOT = Path("/kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset")
SEG_IMAGES_LABELS_ROOT = SEG_ROOT / "vehide_seg"
SEG_DATA_YAML_SOURCE = SEG_ROOT / "damage-seg.yaml"

assert SEG_ROOT.exists(), "Segmentation dataset not found at this path -- check /kaggle/input/ above and update SEG_ROOT."
assert SEG_IMAGES_LABELS_ROOT.exists(), f"Expected images/labels under {SEG_IMAGES_LABELS_ROOT}, not found."
assert SEG_DATA_YAML_SOURCE.exists(), f"Expected damage-seg.yaml at {SEG_DATA_YAML_SOURCE}, not found."

print(open(SEG_DATA_YAML_SOURCE).read())

CLASS_NAMES = ["dent", "scratch", "crack", "broken_lamp", "shattered_glass", "flat_tyre"]

train_n = len(list((SEG_IMAGES_LABELS_ROOT / "images" / "train").glob("*.jpg")))
val_n = len(list((SEG_IMAGES_LABELS_ROOT / "images" / "val").glob("*.jpg")))
print(f"Train images: {train_n}, Val images: {val_n}")

datasets
names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/working/vehide_seg
test: images/test
train: images/train
val: images/val

Train images: 9545, Val images: 2047


In [3]:
import yaml

with open(SEG_DATA_YAML_SOURCE) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(SEG_IMAGES_LABELS_ROOT)

SEG_DATA_YAML = "/kaggle/working/damage-seg.yaml"
with open(SEG_DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)
print(open(SEG_DATA_YAML).read())

names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg
test: images/test
train: images/train
val: images/val



## 3. Fetch the starting checkpoint

Downloads `best.pt` from the earlier continuation run's backup dataset — not assumed to exist locally, since `/kaggle/working/` does not persist across Kaggle sessions.

In [5]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

auth_check = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "zzz_auth_check_zzz", "-p", "1"],
    capture_output=True, text=True,
)
if auth_check.returncode != 0:
    print("STDOUT:", auth_check.stdout)
    print("STDERR:", auth_check.stderr)
    raise RuntimeError(
        "Kaggle API authentication failed. Check Add-ons -> Secrets for "
        "KAGGLE_USERNAME / KAGGLE_KEY, both added AND toggled ON for this notebook."
    )
print("Kaggle API authentication OK.")

# EDIT if the earlier continuation run's backup dataset has a different slug
PRIOR_BACKUP_DATASET_ID = f"{os.environ['KAGGLE_USERNAME']}/cardd-seg-continuation-checkpoint-backup"

os.makedirs("/kaggle/working/prior_checkpoint", exist_ok=True)
dl = subprocess.run(
    ["kaggle", "datasets", "download", PRIOR_BACKUP_DATASET_ID,
     "-p", "/kaggle/working/prior_checkpoint", "--unzip", "-q"],
    capture_output=True, text=True,
)
if dl.returncode != 0:
    print(dl.stdout, dl.stderr)
    raise RuntimeError(
        f"Could not download {PRIOR_BACKUP_DATASET_ID}. Check the slug matches "
        f"your actual earlier run's backup dataset name."
    )

starting_checkpoint_candidates = list(Path("/kaggle/working/prior_checkpoint").rglob("last.pt"))
assert starting_checkpoint_candidates, "No last.pt found in the downloaded backup."
STARTING_CHECKPOINT = str(starting_checkpoint_candidates[0])
print(f"Starting checkpoint: {STARTING_CHECKPOINT}")

Kaggle API authentication OK.
Starting checkpoint: /kaggle/working/prior_checkpoint/last.pt


In [6]:
from ultralytics import YOLO

probe_model = YOLO(STARTING_CHECKPOINT)
print("Checkpoint's own class names:", probe_model.names)
print("Task:", probe_model.task)
n_params = sum(p.numel() for p in probe_model.model.parameters())
print(f"Parameters: {n_params/1e6:.1f}M")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Checkpoint's own class names: {0: 'dent', 1: 'scratch', 2: 'crack', 3: 'broken_lamp', 4: 'shattered_glass', 5: 'flat_tyre'}
Task: segment
Parameters: 11.8M


## 4. Configuration

`cls=0.3` / `dfl=1.7` is the new element this round — not tried on this model lineage before. `lr0` kept low, since this checkpoint is already well-adapted to VehiDE (two rounds of fine-tuning deep), so only small further updates are appropriate.

In [7]:
est_cache_ram_gb = train_n * (1280 * 1280 * 3) / 1e9
print(f"Estimated RAM cost of cache='ram' for the training split: ~{est_cache_ram_gb:.1f} GB")
print(f"Available RAM: {ram_available_gb:.1f} GB")

if est_cache_ram_gb < ram_available_gb * 0.6:
    CACHE_MODE = True
    print("-> Using cache=True (RAM). Comfortable headroom available.")
else:
    CACHE_MODE = "disk"
    print("-> RAM caching looks risky at this dataset size vs. available RAM. Using cache='disk' instead.")

Estimated RAM cost of cache='ram' for the training split: ~46.9 GB
Available RAM: 31.8 GB
-> RAM caching looks risky at this dataset size vs. available RAM. Using cache='disk' instead.


In [8]:
RUN_NAME = "cardd_yolov8s_seg_dfl_boundary"

SESSION_EPOCH_BUDGET = 999
BACKUP_EVERY_N_EPOCHS = 5
BACKUP_SLUG = "cardd-yolov8s-dfl-boundary-checkpoint-backup"   # distinct from every prior run's identity
BACKUP_DATASET_ID = f"{os.environ['KAGGLE_USERNAME']}/{BACKUP_SLUG}"

BACKUP_STAGE = Path("/kaggle/working/checkpoint_backup_dfl_boundary")
BACKUP_STAGE.mkdir(parents=True, exist_ok=True)

TRAIN_ARGS = dict(
    data=SEG_DATA_YAML,
    epochs=30,
    imgsz=1280,
    batch=4,
    optimizer="AdamW",
    lr0=0.0002,
    lrf=0.001,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=2,     # short -- this checkpoint is already well-adapted, not a fresh transition
    cls=0.3,             # new this round -- was 0.5 in both earlier cardd-yolov8s runs
    dfl=1.7,              # new this round -- was left at Ultralytics' default (1.5) in both earlier runs
    patience=15,
    save_period=10,
    amp=True,
    cache=CACHE_MODE,
    workers=4,
    multi_scale=False,   # kept off -- proven OOM contributor on a larger model; no reason to risk it here either
    close_mosaic=10,
    seed=42,
    deterministic=True,
    plots=True,
    project="/kaggle/working/runs/cardd_yolov8s_dfl_boundary",
)
print(json.dumps(TRAIN_ARGS, indent=2))

{
  "data": "/kaggle/working/damage-seg.yaml",
  "epochs": 30,
  "imgsz": 1280,
  "batch": 4,
  "optimizer": "AdamW",
  "lr0": 0.0002,
  "lrf": 0.001,
  "cos_lr": true,
  "weight_decay": 0.0005,
  "warmup_epochs": 2,
  "cls": 0.3,
  "dfl": 1.7,
  "patience": 15,
  "save_period": 10,
  "amp": true,
  "cache": "disk",
  "workers": 4,
  "multi_scale": false,
  "close_mosaic": 10,
  "seed": 42,
  "deterministic": true,
  "plots": true,
  "project": "/kaggle/working/runs/cardd_yolov8s_dfl_boundary"
}


## 5. Probe first — corrected two-part timing measurement

Training and validation timed **separately**. The old combined-measurement approach silently inflated estimates by multiplying a fixed validation cost by the training-set-size scale factor — confirmed directly on a real run (reported ~177 min/epoch when the true cost was ~21 min/epoch).

In [9]:
import random

PROBE_DIR = Path("/kaggle/working/probe_subsample")
probe_img_dir = PROBE_DIR / "images" / "train"
probe_lbl_dir = PROBE_DIR / "labels" / "train"
probe_img_dir.mkdir(parents=True, exist_ok=True)
probe_lbl_dir.mkdir(parents=True, exist_ok=True)

all_train_imgs = sorted((SEG_IMAGES_LABELS_ROOT / "images" / "train").glob("*.jpg"))
sample = random.Random(42).sample(all_train_imgs, min(300, len(all_train_imgs)))
for img_path in sample:
    (probe_img_dir / img_path.name).symlink_to(img_path)
    lbl_path = SEG_IMAGES_LABELS_ROOT / "labels" / "train" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        (probe_lbl_dir / lbl_path.name).symlink_to(lbl_path)

probe_cfg = dict(cfg)
probe_cfg["path"] = str(PROBE_DIR)
probe_cfg["train"] = "images/train"
probe_cfg["val"] = str(SEG_IMAGES_LABELS_ROOT / "images" / "val")

PROBE_YAML = "/kaggle/working/damage-seg-probe.yaml"
with open(PROBE_YAML, "w") as f:
    yaml.safe_dump(probe_cfg, f)
print(f"Probe subsample: {len(sample)} images")

Probe subsample: 300 images


In [10]:
# Two SEPARATE timed measurements -- training-only (val=False) on the
# subsample, then one real validation pass on the full val set -- combined
# correctly rather than scaling a combined number.
train_only_args = dict(TRAIN_ARGS)
train_only_args["data"] = PROBE_YAML
train_only_args["epochs"] = 1
train_only_args["val"] = False
train_only_args["plots"] = False
train_only_args["project"] = "/kaggle/working/runs/probe_dfl_boundary"
train_only_args["name"] = "timing_probe_train_only"
train_only_args["cache"] = False

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
probe_train_model = YOLO(STARTING_CHECKPOINT)
probe_train_model.train(**train_only_args)
train_only_wall = time.time() - t0
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Training-only time (300 images, no validation): {train_only_wall/60:.1f} min")
print(f"Peak VRAM used during training: {peak_vram_gb:.2f} GB / {total_vram_gb:.1f} GB "
      f"({100*peak_vram_gb/total_vram_gb:.0f}%)")

if peak_vram_gb > total_vram_gb * 0.85:
    print("\nWARNING: peak VRAM usage is close to the card's limit even on this "
          "small probe. Lower TRAIN_ARGS['imgsz'] or ['batch'] in Section 4 and "
          "re-run this probe before continuing to Section 6.")
else:
    print("VRAM headroom looks OK for this config.")

t0 = time.time()
_ = probe_train_model.val(
    data=SEG_DATA_YAML, split="val",
    imgsz=TRAIN_ARGS["imgsz"], batch=TRAIN_ARGS["batch"], verbose=False,
)
val_only_wall = time.time() - t0
print(f"\nValidation time on the real, full val set ({val_n} images): {val_only_wall/60:.1f} min")

train_scale_factor = train_n / len(sample)
est_train_time_full_min = (train_only_wall / 60) * train_scale_factor
est_val_time_min = val_only_wall / 60
full_epoch_est_min = est_train_time_full_min + est_val_time_min

print(f"\nEstimated training time per epoch on the full {train_n}-image set: "
      f"{est_train_time_full_min:.1f} min")
print(f"Estimated validation time per epoch (fixed): {est_val_time_min:.1f} min")
print(f"Corrected estimated total per epoch: {full_epoch_est_min:.1f} min")
print(f"Corrected estimated total for TRAIN_ARGS['epochs']={TRAIN_ARGS['epochs']}: "
      f"{full_epoch_est_min * TRAIN_ARGS['epochs'] / 60:.1f} hours")
print("\nIf that total is impractical against your remaining Kaggle GPU quota, "
      "lower TRAIN_ARGS['epochs'] in Section 4 now and re-run that cell before "
      "continuing to Section 6.")

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.3, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/damage-seg-probe.yaml, degrees=0.0, deterministic=True, device=, dfl=1.7, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/prior_checkpoint/last.pt, momentum=0.937, mosaic=1.0, multi_scale=Fal

## 6. Fine-tune, with multi-session checkpoint relay

In [11]:
meta_check = subprocess.run(
    ["kaggle", "datasets", "metadata", BACKUP_DATASET_ID, "-p", "/tmp/meta_check_dfl_boundary"],
    capture_output=True, text=True,
)
check = meta_check.returncode
resume_from = None

if check == 0:
    print(f"Found existing backup: {BACKUP_DATASET_ID} -- downloading...")
    os.makedirs("/kaggle/working/recovered_dfl_boundary", exist_ok=True)
    dl2 = subprocess.run(
        ["kaggle", "datasets", "download", BACKUP_DATASET_ID,
         "-p", "/kaggle/working/recovered_dfl_boundary", "--unzip", "-q"],
        capture_output=True, text=True,
    )
    if dl2.returncode != 0:
        print(dl2.stdout, dl2.stderr)
        raise RuntimeError("Backup dataset metadata was found, but downloading it failed.")
    recovered_pt = list(Path("/kaggle/working/recovered_dfl_boundary").rglob("last.pt"))
    if recovered_pt:
        resume_from = recovered_pt[0]
        epoch_marker = Path("/kaggle/working/recovered_dfl_boundary/epoch.txt")
        print(f"Will resume from epoch {epoch_marker.read_text().strip() if epoch_marker.exists() else '?'}")
else:
    print(f"No existing backup dataset ({BACKUP_DATASET_ID}) -- session 1, starting fresh from {STARTING_CHECKPOINT}")
    print(f"(kaggle CLI returned: {meta_check.stderr.strip()[:200]})")

No existing backup dataset (m4rcuseryx/cardd-yolov8s-dfl-boundary-checkpoint-backup) -- session 1, starting fresh from /kaggle/working/prior_checkpoint/last.pt
(kaggle CLI returned: )


In [12]:
_session_start_epoch = {"value": None}
_dataset_exists = {"value": check == 0}
_backup_failures = []


def _push_backup(trainer, completed_epoch):
    last_pt = trainer.save_dir / "weights" / "last.pt"
    if not last_pt.exists():
        return
    shutil.copy(last_pt, BACKUP_STAGE / "last.pt")
    for extra in ("results.csv", "args.yaml"):
        p = trainer.save_dir / extra
        if p.exists():
            shutil.copy(p, BACKUP_STAGE / extra)
    (BACKUP_STAGE / "epoch.txt").write_text(str(completed_epoch))
    (BACKUP_STAGE / "dataset-metadata.json").write_text(json.dumps({
        "title": "cardd-yolov8s DFL boundary-precision checkpoint backup",
        "id": BACKUP_DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }))

    if not _dataset_exists["value"]:
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(BACKUP_STAGE), "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            _dataset_exists["value"] = True
    else:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(BACKUP_STAGE),
             "-m", f"epoch {completed_epoch}", "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )

    if result.returncode == 0:
        print(f"Backed up checkpoint at epoch {completed_epoch} -> {BACKUP_DATASET_ID}")
    else:
        msg = f"BACKUP FAILED at epoch {completed_epoch}: {result.stderr.strip()[:300]}"
        print(f"\n{'='*70}\n{msg}\n{'='*70}\n")
        _backup_failures.append((completed_epoch, result.stderr.strip()))


def relay_callback(trainer):
    completed = trainer.epoch + 1
    if _session_start_epoch["value"] is None:
        _session_start_epoch["value"] = completed - 1
    epochs_this_session = completed - _session_start_epoch["value"]
    is_budget_stop = epochs_this_session >= SESSION_EPOCH_BUDGET
    is_periodic_backup = completed % BACKUP_EVERY_N_EPOCHS == 0

    if is_budget_stop or is_periodic_backup:
        _push_backup(trainer, completed)
    if is_budget_stop:
        print(f"\nSession budget reached (total completed: {completed}/{trainer.epochs}). Stopping gracefully.")
        trainer.stop = True


t0 = time.time()
if resume_from is not None:
    model = YOLO(str(resume_from))
    model.add_callback("on_train_epoch_end", relay_callback)
    results = model.train(resume=True)
else:
    model = YOLO(STARTING_CHECKPOINT)
    model.add_callback("on_train_epoch_end", relay_callback)
    results = model.train(name=RUN_NAME, **TRAIN_ARGS)
wall = time.time() - t0
print(f"\nThis session: {wall/60:.1f} min")

if _backup_failures:
    print(f"\nWARNING: {len(_backup_failures)} backup attempt(s) failed. "
          f"Failed epochs: {[e for e, _ in _backup_failures]}")
else:
    print("\nAll backup attempts this session succeeded.")

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.3, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/damage-seg.yaml, degrees=0.0, deterministic=True, device=, dfl=1.7, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/prior_checkpoint/last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, na

## 7. Evaluate (same per-class format, for direct comparison against both earlier `cardd-yolov8s` runs)

In [14]:
m = model.val(data=SEG_DATA_YAML, split="test", imgsz=TRAIN_ARGS["imgsz"])
print("Box    mAP50:", float(m.box.map50), " mAP50-95:", float(m.box.map))
print("Mask   mAP50:", float(m.seg.map50), " mAP50-95:", float(m.seg.map))

import pandas as pd
rows = []
for idx, ci in enumerate(m.box.ap_class_index):
    rows.append({
        "class": CLASS_NAMES[int(ci)],
        "box_mAP50": float(m.box.ap50[idx]),
        "mask_mAP50": float(m.seg.ap50[idx]),
    })
pd.DataFrame(rows).sort_values("mask_mAP50", ascending=False)

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 485.7±199.4 MB/s, size: 310.6 KB)
val: Scanning /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg/labels/test... 2047 images, 140 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2047/2047 986.9it/s 2.1s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 128/128 1.1s/it 2:211.1ss
                   all       2047       4846      0.481      0.425      0.386        0.2       0.47      0.403      0.355      0.166
                  dent        648        825      0.429      0.278      0.254      0.108      0.419      0.261      0.243     0.0961
               scratch       1007       2174      0.

,class,box_mAP50,mask_mAP50
4,shattered_glass,0.637035,0.602120
5,flat_tyre,0.483673,0.490469
3,broken_lamp,0.460535,0.376608
0,dent,0.253626,0.243209
2,crack,0.222528,0.208497
1,scratch,0.258428,0.208477
